In [1]:
!pip install langchain langchain-groq langchain-huggingface chromadb

In [2]:
import os
os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"

In [3]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

# Connect to LLM
llm = ChatGroq(model="llama-3.1-8b-instant")

# Ask a question
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What is machine learning in 2 sentences?")
]

response = llm.invoke(messages)
print(response.content)

Machine learning is a subset of artificial intelligence that involves training algorithms to learn from data, enabling them to make predictions or decisions without being explicitly programmed. By analyzing patterns, relationships, and trends in data, machine learning models can improve their accuracy over time, allowing them to adapt to new information and make informed decisions.


In [4]:
pip install langchain-community

In [5]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Step 1: Create documents
documents = [
    Document(page_content="The company refund policy allows returns within 30 days of purchase."),
    Document(page_content="To get a refund, customers must provide their order number and reason for return."),
    Document(page_content="Refunds are processed within 5-7 business days after approval."),
    Document(page_content="Products must be in original condition to qualify for a refund."),
    Document(page_content="Digital products and gift cards are not eligible for refunds.")
]

# Step 2: Create embeddings and store in Vector DB
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents, embeddings)

print("Documents stored successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Documents stored successfully!


In [6]:
from langchain_core.prompts import ChatPromptTemplate

# Step 1: Set up retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Step 2: Create prompt template
prompt = ChatPromptTemplate.from_template("""
Answer the question based ONLY on the following context:

{context}

Question: {question}
""")

# Step 3: Ask a question
user_question = "How long does a refund take?"

# Step 4: Retrieve relevant documents
retrieved_docs = retriever.invoke(user_question)
context = "\n".join([doc.page_content for doc in retrieved_docs])

# Step 5: Generate answer
chain = prompt | llm
response = chain.invoke({
    "context": context,
    "question": user_question
})

print("Question:", user_question)
print("\nAnswer:", response.content)

Question: How long does a refund take?

Answer: A refund takes 5-7 business days after approval.
